# 📚 BiblioRec — Sistema de Recomendação de Livros Técnicos

Implementação completa no Google Colab com motor híbrido (TF-IDF + KNN) e interfaces Gradio.

In [1]:
# @title 📦 Instalar dependências
!pip install -q gradio pandas scikit-learn sqlalchemy

In [2]:
# @title 🔧 Importações e configuração
import sqlite3
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from datetime import datetime
import hashlib
import warnings
warnings.filterwarnings('ignore')

DB_PATH = "/content/bibliorec.db"

print("✅ Ambiente configurado")
print(f"📁 Banco de dados: {DB_PATH}")

✅ Ambiente configurado
📁 Banco de dados: /content/bibliorec.db


In [3]:
# @title 🗄️ Criar banco de dados SQLite
def criar_banco():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            senha_hash TEXT NOT NULL,
            curso TEXT,
            role TEXT DEFAULT 'aluno',
            criado_em TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS livros (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            autor TEXT,
            area TEXT,
            isbn TEXT UNIQUE,
            palavras_chave TEXT,
            quantidade_total INTEGER DEFAULT 1,
            quantidade_disponivel INTEGER DEFAULT 1
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS emprestimos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            aluno_id INTEGER,
            livro_id INTEGER,
            data_emprestimo TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            data_devolucao TIMESTAMP,
            status TEXT DEFAULT 'ativo',
            FOREIGN KEY (aluno_id) REFERENCES usuarios(id),
            FOREIGN KEY (livro_id) REFERENCES livros(id)
        )
    """)

    conn.commit()
    conn.close()
    print("✅ Tabelas criadas com sucesso")

def hash_senha(senha):
    return hashlib.sha256(senha.encode()).hexdigest()

criar_banco()

✅ Tabelas criadas com sucesso


In [4]:
# @title 🌱 Popular banco com dados de exemplo
def popular_dados():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("DELETE FROM emprestimos")
    cur.execute("DELETE FROM livros")
    cur.execute("DELETE FROM usuarios")

    usuarios = [
        ("ADM", "ADM@teste.com", hash_senha("123456"), "Ciência da Computação", "aluno"),
        ("Bruno Costa", "bruno@teste.com", hash_senha("123456"), "Ciência da Computação", "aluno"),
        ("Carla Dias", "carla@teste.com", hash_senha("123456"), "Engenharia Civil", "aluno"),
        ("Diego Rocha", "diego@teste.com", hash_senha("123456"), "Administração", "aluno"),
        ("Admin BiblioRec", "admin@bibliorec.com", hash_senha("admin123"), None, "admin"),
    ]
    cur.executemany(
        "INSERT INTO usuarios (nome, email, senha_hash, curso, role) VALUES (?, ?, ?, ?, ?)",
        usuarios
    )

    livros = [
        ("Algoritmos: Teoria e Prática", "Cormen", "Computação", "978-85-3521-258-1", "algoritmos programação estrutura dados"),
        ("Estruturas de Dados e Algoritmos em Python", "Goodrich", "Computação", "978-85-5081-206-6", "estrutura dados python algoritmos"),
        ("Introdução à Programação com Python", "Nilo Ney", "Computação", "978-85-7522-718-3", "programação python iniciante"),
        ("Banco de Dados: Teoria e Desenvolvimento", "Silberschatz", "Computação", "978-85-2161-743-2", "banco dados sql modelagem"),
        ("Redes de Computadores", "Tanenbaum", "Computação", "978-85-4300-807-4", "redes protocolos internet"),
        ("Inteligência Artificial", "Russell & Norvig", "Computação", "978-85-3521-763-0", "ia machine learning inteligência"),
        ("Cálculo: Volume 1", "Stewart", "Matemática", "978-85-2211-428-0", "cálculo matemática limites"),
        ("Álgebra Linear", "Anton", "Matemática", "978-85-2161-540-7", "álgebra linear matrizes"),
        ("Resistência dos Materiais", "Hibbeler", "Engenharia Civil", "978-85-4300-336-9", "estruturas materiais resistência"),
        ("Estruturas de Concreto Armado", "Araújo", "Engenharia Civil", "978-85-7183-822-3", "concreto estruturas civil"),
        ("Mecânica dos Solos", "Caputo", "Engenharia Civil", "978-85-2161-742-5", "solos fundações geotecnia"),
        ("Gestão de Projetos", "Kerzner", "Administração", "978-85-7608-152-4", "gestão projetos pmi"),
        ("Marketing Digital", "Kotler", "Administração", "978-85-4300-104-4", "marketing digital estratégia"),
        ("Finanças Corporativas", "Brealey", "Administração", "978-85-7722-082-3", "finanças investimento gestão"),
        ("Estatística Aplicada", "Bussab", "Estatística", "978-85-2153-326-8", "estatística dados probabilidade"),
    ]
    cur.executemany(
        """INSERT INTO livros (titulo, autor, area, isbn, palavras_chave)
           VALUES (?, ?, ?, ?, ?)""",
        livros
    )

    emprestimos = [
        (1, 1), (1, 2), (1, 3),
        (2, 1), (2, 4), (2, 5),
        (3, 9), (3, 10),
        (4, 12), (4, 13),
        (1, 6), (2, 6),
    ]
    cur.executemany(
        "INSERT INTO emprestimos (aluno_id, livro_id) VALUES (?, ?)",
        emprestimos
    )

    conn.commit()
    conn.close()
    print("✅ Dados de exemplo inseridos")
    print(f"   → {len(usuarios)} usuários")
    print(f"   → {len(livros)} livros")
    print(f"   → {len(emprestimos)} empréstimos")

popular_dados()

conn = sqlite3.connect(DB_PATH)
print("\n📚 Amostra do acervo:")
display(pd.read_sql("SELECT id, titulo, area FROM livros LIMIT 5", conn))
conn.close()

✅ Dados de exemplo inseridos
   → 5 usuários
   → 15 livros
   → 12 empréstimos

📚 Amostra do acervo:


,id,titulo,area
0,1,Algoritmos: Teoria e Prática,Computação
1,2,Estruturas de Dados e Algoritmos em Python,Computação
2,3,Introdução à Programação com Python,Computação
3,4,Banco de Dados: Teoria e Desenvolvimento,Computação
4,5,Redes de Computadores,Computação


In [5]:
# @title 🧠 Motor de recomendação híbrido (CORRIGIDO)
class BiblioRec:
    def __init__(self, db_path):
        self.db_path = db_path
        self.livros_df = None
        self.matriz_tfidf = None
        self.tfidf = None
        self.matriz_emprestimos = None
        self.alunos_idx = None
        self.livros_idx = None
        self._treinar()

    def _carregar_livros(self):
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql("SELECT * FROM livros", conn)
        conn.close()
        return df

    def _treinar(self):
        # --- TF-IDF ---
        self.livros_df = self._carregar_livros()
        self.livros_df['perfil'] = (
            self.livros_df['titulo'] + ' ' +
            self.livros_df['area'] + ' ' +
            self.livros_df['palavras_chave'].fillna('')
        )
        self.tfidf = TfidfVectorizer(stop_words=None, lowercase=True)
        self.matriz_tfidf = self.tfidf.fit_transform(self.livros_df['perfil'])

        # --- Matriz aluno × livro (CORRIGIDA) ---
        conn = sqlite3.connect(self.db_path)
        hist = pd.read_sql(
            "SELECT aluno_id, livro_id FROM emprestimos WHERE status='ativo'",
            conn
        )
        conn.close()

        if len(hist) > 0:
            # Usar crosstab é mais robusto que pivot_table com values duplicados
            matriz_df = pd.crosstab(hist['aluno_id'], hist['livro_id'])
            # Garantir que TODOS os livros apareçam como colunas
            todos_livros = self.livros_df['id'].tolist()
            for lid in todos_livros:
                if lid not in matriz_df.columns:
                    matriz_df[lid] = 0
            matriz_df = matriz_df[sorted(matriz_df.columns)]

            self.matriz_emprestimos = csr_matrix(matriz_df.values)
            self.alunos_idx = list(matriz_df.index)
            self.livros_idx = list(matriz_df.columns)

            print(f"📊 Matriz de empréstimos: {self.matriz_emprestimos.shape} "
                  f"({len(self.alunos_idx)} alunos × {len(self.livros_idx)} livros)")
        else:
            self.matriz_emprestimos = None
            print("⚠️ Sem empréstimos registrados")

        print(f"🧠 Modelo treinado com {len(self.livros_df)} livros")

    def _recomendar_conteudo(self, curso, top_n=20):
        mapa_cursos = {
            'Ciência da Computação': 'algoritmos programação estrutura dados python banco redes inteligência',
            'Engenharia Civil': 'estruturas materiais resistência concreto solos civil',
            'Administração': 'gestão projetos marketing finanças estratégia',
            'Matemática': 'cálculo álgebra matemática estatística',
            'Estatística': 'estatística dados probabilidade cálculo',
        }
        termos = mapa_cursos.get(curso, '')
        if not termos:
            return self.livros_df.head(top_n)

        vetor = self.tfidf.transform([termos])
        sims = cosine_similarity(vetor, self.matriz_tfidf).flatten()
        idx = sims.argsort()[-top_n:][::-1]
        return self.livros_df.iloc[idx]

    def _recomendar_colaborativo(self, aluno_id, top_n=20):
        if self.matriz_emprestimos is None or aluno_id not in self.alunos_idx:
            return []

        # Garantir que haja pelo menos 1 feature
        if self.matriz_emprestimos.shape[1] < 1:
            return []

        modelo = NearestNeighbors(metric='cosine', algorithm='brute')
        modelo.fit(self.matriz_emprestimos)

        pos = self.alunos_idx.index(aluno_id)
        k = min(4, len(self.alunos_idx))
        dist, indices = modelo.kneighbors(
            self.matriz_emprestimos[pos], n_neighbors=k
        )

        lidos = set(
            self.livros_idx[i]
            for i, v in enumerate(self.matriz_emprestimos[pos].toarray()[0])
            if v > 0
        )

        candidatos = {}
        for idx in indices.flatten()[1:]:
            vetor = self.matriz_emprestimos[idx].toarray()[0]
            for i, v in enumerate(vetor):
                if v > 0 and self.livros_idx[i] not in lidos:
                    candidatos[self.livros_idx[i]] = candidatos.get(self.livros_idx[i], 0) + 1

        return sorted(candidatos, key=candidatos.get, reverse=True)[:top_n]

    def recomendar(self, aluno_id, curso, top_n=10):
        conteudo = self._recomendar_conteudo(curso, top_n=20)
        colaborativo = self._recomendar_colaborativo(aluno_id, top_n=20)

        scores = {}
        for i, row in enumerate(conteudo.itertuples()):
            scores[row.id] = scores.get(row.id, 0) + 0.6 * (1 - i / max(len(conteudo), 1))

        for i, livro_id in enumerate(colaborativo):
            scores[livro_id] = scores.get(livro_id, 0) + 0.4 * (1 - i / max(len(colaborativo), 1))

        ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
        ids = [r[0] for r in ranking]

        if not ids:
            return pd.DataFrame()

        resultado = self.livros_df[self.livros_df['id'].isin(ids)].copy()
        resultado['score'] = resultado['id'].map(dict(ranking))
        resultado = resultado.sort_values('score', ascending=False)
        return resultado[['id', 'titulo', 'autor', 'area', 'quantidade_disponivel', 'score']]


# Instanciar motor (fora da classe)
motor = BiblioRec(DB_PATH)

# Teste rápido
print("\n🔎 Teste de recomendação (Ana, Ciência da Computação):")
display(motor.recomendar(aluno_id=1, curso='Ciência da Computação', top_n=5))

📊 Matriz de empréstimos: (4, 15) (4 alunos × 15 livros)
🧠 Modelo treinado com 15 livros

🔎 Teste de recomendação (Ana, Ciência da Computação):


,id,titulo,autor,area,quantidade_disponivel,score
3,4,Banco de Dados: Teoria e Desenvolvimento,Silberschatz,Computação,1,0.880000
4,5,Redes de Computadores,Tanenbaum,Computação,1,0.773333
1,2,Estruturas de Dados e Algoritmos em Python,Goodrich,Computação,1,0.600000
8,9,Resistência dos Materiais,Hibbeler,Engenharia Civil,1,0.586667
0,1,Algoritmos: Teoria e Prática,Cormen,Computação,1,0.560000


In [6]:
# @title 🛠️ Funções do sistema
def autenticar(email, senha):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(
        "SELECT id, nome, curso, role FROM usuarios WHERE email=? AND senha_hash=?",
        conn, params=(email, hash_senha(senha))
    )
    conn.close()
    if df.empty:
        return None
    return df.iloc[0].to_dict()

def buscar_livros(termo):
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT id, titulo, autor, area, quantidade_disponivel
        FROM livros
        WHERE titulo LIKE ? OR autor LIKE ? OR area LIKE ? OR palavras_chave LIKE ?
    """
    t = f"%{termo}%"
    df = pd.read_sql(query, conn, params=(t, t, t, t))
    conn.close()
    return df

def cadastrar_livro(titulo, autor, area, isbn, palavras_chave, qtd):
    try:
        conn = sqlite3.connect(DB_PATH)
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO livros (titulo, autor, area, isbn, palavras_chave, quantidade_total, quantidade_disponivel)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (titulo, autor, area, isbn, palavras_chave, qtd, qtd))
        conn.commit()
        conn.close()
        motor._treinar()
        return f"✅ Livro '{titulo}' cadastrado com sucesso!"
    except Exception as e:
        return f"❌ Erro: {e}"

def listar_emprestimos():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT e.id, u.nome AS aluno, l.titulo AS livro, e.data_emprestimo, e.status
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros l ON l.id = e.livro_id
        ORDER BY e.data_emprestimo DESC
    """, conn)
    conn.close()
    return df

print("✅ Funções do sistema carregadas")

✅ Funções do sistema carregadas


In [7]:
# @title 🆕 Funções de cadastro e empréstimo
import re

def validar_email(email):
    return re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", email) is not None

def cadastrar_usuario(nome, email, senha, curso):
    """Cadastra um novo aluno no banco."""
    # Validações
    if not nome or not nome.strip():
        return "❌ Informe seu nome completo."
    if not validar_email(email):
        return "❌ E-mail inválido."
    if len(senha) < 6:
        return "❌ A senha deve ter pelo menos 6 caracteres."
    if not curso or curso == "—":
        return "❌ Selecione seu curso."

    try:
        conn = sqlite3.connect(DB_PATH)
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO usuarios (nome, email, senha_hash, curso, role)
            VALUES (?, ?, ?, ?, 'aluno')
        """, (nome.strip(), email.strip().lower(), hash_senha(senha), curso))
        conn.commit()
        conn.close()
        return f"✅ Conta criada com sucesso! Faça login com {email}."
    except sqlite3.IntegrityError:
        return "❌ Este e-mail já está cadastrado. Tente fazer login."
    except Exception as e:
        return f"❌ Erro: {e}"


def registrar_emprestimo(aluno_id, livro_id):
    """Registra empréstimo e desconta do estoque."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # Verifica disponibilidade
    cur.execute("SELECT titulo, quantidade_disponivel FROM livros WHERE id=?", (livro_id,))
    r = cur.fetchone()
    if not r:
        conn.close()
        return "❌ Livro não encontrado."
    titulo, disponivel = r
    if disponivel <= 0:
        conn.close()
        return f"❌ '{titulo}' está indisponível no momento."

    # Verifica se o aluno já tem esse livro ativo
    cur.execute("""
        SELECT id FROM emprestimos
        WHERE aluno_id=? AND livro_id=? AND status='ativo'
    """, (aluno_id, livro_id))
    if cur.fetchone():
        conn.close()
        return f"⚠️ Você já está com '{titulo}' emprestado."

    # Registra e atualiza estoque
    cur.execute("INSERT INTO emprestimos (aluno_id, livro_id) VALUES (?, ?)",
                (aluno_id, livro_id))
    cur.execute("""
        UPDATE livros SET quantidade_disponivel = quantidade_disponivel - 1
        WHERE id=?
    """, (livro_id,))
    conn.commit()
    conn.close()

    motor._treinar()  # retreina com o novo empréstimo
    return f"✅ Empréstimo de '{titulo}' registrado com sucesso!"


def devolver_livro(aluno_id, titulo):
    """Marca empréstimo como devolvido e devolve ao estoque."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        SELECT e.id, e.livro_id FROM emprestimos e
        JOIN livros l ON l.id = e.livro_id
        WHERE e.aluno_id=? AND l.titulo=? AND e.status='ativo'
        LIMIT 1
    """, (aluno_id, titulo))
    r = cur.fetchone()
    if not r:
        conn.close()
        return f"❌ Você não tem empréstimo ativo de '{titulo}'."

    emp_id, livro_id = r
    cur.execute("""
        UPDATE emprestimos SET status='devolvido', data_devolucao=CURRENT_TIMESTAMP
        WHERE id=?
    """, (emp_id,))
    cur.execute("""
        UPDATE livros SET quantidade_disponivel = quantidade_disponivel + 1
        WHERE id=?
    """, (livro_id,))
    conn.commit()
    conn.close()
    motor._treinar()
    return f"✅ '{titulo}' devolvido com sucesso!"


def listar_cursos():
    """Lista de cursos disponíveis para o cadastro."""
    return [
        "Ciência da Computação",
        "Engenharia Civil",
        "Administração",
        "Matemática",
        "Estatística",
        "Outro",
    ]

print("✅ Funções de cadastro e empréstimo carregadas")

✅ Funções de cadastro e empréstimo carregadas


In [8]:
# @title 🎓 Interface do Aluno (com estado por sessão)
import gradio as gr
import sqlite3
import pandas as pd

# ---------- FUNÇÕES ----------
def _dados_aluno(uid, nome, curso):
    """Carrega recomendações, histórico e dropdowns do aluno."""
    if not uid:
        return (pd.DataFrame(), pd.DataFrame(),
                gr.update(choices=[], value=None),
                gr.update(choices=[], value=None))

    rec = motor.recomendar(uid, curso or "", top_n=8)
    if not rec.empty:
        rec = rec[['id', 'titulo', 'autor', 'area', 'quantidade_disponivel']]
    titulos_rec = rec['titulo'].tolist() if not rec.empty else []

    conn = sqlite3.connect(DB_PATH)
    hist = pd.read_sql("""
        SELECT l.titulo AS Livro,
               e.data_emprestimo AS "Emprestado em",
               COALESCE(e.data_devolucao, '—') AS "Devolvido em",
               e.status AS Status
        FROM emprestimos e
        JOIN livros l ON l.id = e.livro_id
        WHERE e.aluno_id = ?
        ORDER BY e.data_emprestimo DESC
    """, conn, params=(uid,))

    ativos = pd.read_sql("""
        SELECT l.titulo FROM emprestimos e
        JOIN livros l ON l.id = e.livro_id
        WHERE e.aluno_id = ? AND e.status='ativo'
    """, conn, params=(uid,))
    conn.close()

    return (
        rec, hist,
        gr.update(choices=titulos_rec, value=None),
        gr.update(choices=ativos['titulo'].tolist(), value=None),
    )


def fazer_login(email, senha):
    user = autenticar(email, senha)
    if not user:
        return (
            None, None, None,
            "❌ Credenciais inválidas.",
            gr.update(visible=True), gr.update(visible=False),
            pd.DataFrame(), pd.DataFrame(),
            gr.update(choices=[], value=None),
            gr.update(choices=[], value=None),
        )
    rec, hist, dd_rec, dd_dev = _dados_aluno(user['id'], user['nome'], user.get('curso'))
    boas = f"👋 Olá, **{user['nome']}**! Curso: *{user.get('curso') or '—'}*"
    return (
        user['id'], user['nome'], user.get('curso'),
        boas,
        gr.update(visible=False), gr.update(visible=True),
        rec, hist, dd_rec, dd_dev,
    )


def fazer_cadastro(nome, email, senha, curso):
    msg = cadastrar_usuario(nome, email, senha, curso)
    if not msg.startswith("✅"):
        return (
            None, None, None, msg,
            gr.update(visible=True), gr.update(visible=False),
            pd.DataFrame(), pd.DataFrame(),
            gr.update(choices=[], value=None),
            gr.update(choices=[], value=None),
        )
    user = autenticar(email, senha)
    rec, hist, dd_rec, dd_dev = _dados_aluno(user['id'], user['nome'], user.get('curso'))
    boas = f"✅ Conta criada! Bem-vindo, **{nome}**."
    return (
        user['id'], user['nome'], user.get('curso'),
        boas,
        gr.update(visible=False), gr.update(visible=True),
        rec, hist, dd_rec, dd_dev,
    )


def pegar_livro(uid, curso, titulo):
    if not uid:
        return "🔒 Faça login primeiro.", pd.DataFrame(), gr.update(), gr.update()
    if not titulo:
        return "⚠️ Selecione um livro.", pd.DataFrame(), gr.update(), gr.update()

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT id FROM livros WHERE titulo=?", (titulo,))
    r = cur.fetchone()
    conn.close()
    if not r:
        return "❌ Livro não encontrado.", pd.DataFrame(), gr.update(), gr.update()

    msg = registrar_emprestimo(uid, r[0])
    _, hist, dd_rec, dd_dev = _dados_aluno(uid, "", curso)
    return msg, hist, dd_rec, dd_dev


def devolver_livro_ui(uid, curso, titulo):
    if not uid:
        return "🔒 Faça login primeiro.", pd.DataFrame(), gr.update(), gr.update()
    if not titulo:
        return "⚠️ Selecione um livro.", pd.DataFrame(), gr.update(), gr.update()

    msg = devolver_livro(uid, titulo)
    _, hist, dd_rec, dd_dev = _dados_aluno(uid, "", curso)
    return msg, hist, dd_rec, dd_dev


def recarregar_aluno(uid, curso):
    _, hist, dd_rec, dd_dev = _dados_aluno(uid, "", curso)
    return hist, dd_rec, dd_dev


def sair():
    return (
        None, None, None,
        "🔒 Você saiu da conta.",
        gr.update(visible=True), gr.update(visible=False),
        pd.DataFrame(), pd.DataFrame(),
        gr.update(choices=[], value=None),
        gr.update(choices=[], value=None),
    )


def listar_cursos():
    return ["Ciência da Computação", "Engenharia Civil", "Administração",
            "Matemática", "Estatística", "Outro"]


# ---------- INTERFACE ----------
with gr.Blocks(theme=gr.themes.Soft(), title="BiblioRec — Aluno") as app_aluno:
    gr.Markdown("# 📚 BiblioRec — Área do Aluno")

    # Estado por sessão
    st_uid = gr.State(None)
    st_nome = gr.State(None)
    st_curso = gr.State(None)

    # ========== AUTH ==========
    with gr.Group(visible=True) as grupo_auth:
        with gr.Tabs():
            with gr.Tab("🔑 Entrar"):
                login_email = gr.Textbox(label="E-mail", value="ana@teste.com")
                login_senha = gr.Textbox(label="Senha", type="password", value="123456")
                btn_login = gr.Button("Entrar", variant="primary")
                msg_login = gr.Markdown()

            with gr.Tab("🆕 Criar conta"):
                cad_nome = gr.Textbox(label="Nome completo")
                cad_email = gr.Textbox(label="E-mail")
                cad_senha = gr.Textbox(label="Senha (mín. 6 caracteres)", type="password")
                cad_curso = gr.Dropdown(label="Curso",
                                        choices=listar_cursos(),
                                        value="Ciência da Computação")
                btn_cad = gr.Button("Criar conta", variant="primary")
                msg_cad = gr.Markdown()

    # ========== ÁREA LOGADA ==========
    with gr.Group(visible=False) as grupo_area:
        boas_vindas = gr.Markdown("👋 Olá!")
        with gr.Row():
            btn_recarregar = gr.Button("🔄 Atualizar", size="sm")
            btn_sair = gr.Button("🚪 Sair", size="sm")

        gr.Markdown("### 🎯 Recomendações personalizadas")
        tabela_rec = gr.Dataframe(interactive=False)

        gr.Markdown("### 📥 Pegar um livro emprestado")
        with gr.Row():
            dropdown_livros = gr.Dropdown(label="Escolha um livro", choices=[])
            btn_pegar = gr.Button("📚 Pegar emprestado", variant="primary")
        msg_emprestimo = gr.Markdown()

        gr.Markdown("### 📖 Meu histórico de empréstimos")
        tabela_hist = gr.Dataframe(interactive=False)

        gr.Markdown("### 🔄 Devolver um livro")
        with gr.Row():
            dropdown_devolver = gr.Dropdown(label="Selecione um livro ativo", choices=[])
            btn_devolver = gr.Button("↩️ Devolver")
        msg_devolucao = gr.Markdown()

    # ========== LIGAÇÕES ==========
    btn_login.click(
        fazer_login,
        inputs=[login_email, login_senha],
        outputs=[st_uid, st_nome, st_curso,
                 boas_vindas, grupo_auth, grupo_area,
                 tabela_rec, tabela_hist, dropdown_livros, dropdown_devolver],
    )

    btn_cad.click(
        fazer_cadastro,
        inputs=[cad_nome, cad_email, cad_senha, cad_curso],
        outputs=[st_uid, st_nome, st_curso,
                 boas_vindas, grupo_auth, grupo_area,
                 tabela_rec, tabela_hist, dropdown_livros, dropdown_devolver],
    )

    btn_pegar.click(
        pegar_livro,
        inputs=[st_uid, st_curso, dropdown_livros],
        outputs=[msg_emprestimo, tabela_hist, dropdown_livros, dropdown_devolver],
    )

    btn_devolver.click(
        devolver_livro_ui,
        inputs=[st_uid, st_curso, dropdown_devolver],
        outputs=[msg_devolucao, tabela_hist, dropdown_livros, dropdown_devolver],
    )

    btn_recarregar.click(
        recarregar_aluno,
        inputs=[st_uid, st_curso],
        outputs=[tabela_hist, dropdown_livros, dropdown_devolver],
    )

    btn_sair.click(
        sair,
        outputs=[st_uid, st_nome, st_curso,
                 boas_vindas, grupo_auth, grupo_area,
                 tabela_rec, tabela_hist, dropdown_livros, dropdown_devolver],
    )

In [9]:
# @title 📊 Funções do Dashboard do Admin
import pandas as pd
import sqlite3

def estatisticas_gerais():
    """Retorna um resumo numérico do sistema."""
    conn = sqlite3.connect(DB_PATH)

    total_alunos = pd.read_sql(
        "SELECT COUNT(*) AS n FROM usuarios WHERE role='aluno'", conn
    )['n'][0]

    total_livros = pd.read_sql(
        "SELECT COUNT(*) AS n FROM livros", conn
    )['n'][0]

    total_exemplares = pd.read_sql(
        "SELECT COALESCE(SUM(quantidade_total),0) AS n FROM livros", conn
    )['n'][0]

    disponiveis = pd.read_sql(
        "SELECT COALESCE(SUM(quantidade_disponivel),0) AS n FROM livros", conn
    )['n'][0]

    emprestados_ativos = pd.read_sql(
        "SELECT COUNT(*) AS n FROM emprestimos WHERE status='ativo'", conn
    )['n'][0]

    devolvidos = pd.read_sql(
        "SELECT COUNT(*) AS n FROM emprestimos WHERE status='devolvido'", conn
    )['n'][0]

    conn.close()

    resumo = f"""
### 📈 Resumo Geral do Sistema

| Métrica | Valor |
|---|---|
| 👥 Alunos cadastrados | **{total_alunos}** |
| 📚 Títulos no acervo | **{total_livros}** |
| 📦 Exemplares totais | **{total_exemplares}** |
| ✅ Exemplares disponíveis | **{disponiveis}** |
| 📕 Exemplares emprestados | **{emprestados_ativos}** |
| 📗 Empréstimos devolvidos | **{devolvidos}** |
"""
    return resumo


def listar_usuarios():
    """Lista todos os usuários cadastrados com contagem de empréstimos."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT
            u.id,
            u.nome,
            u.email,
            COALESCE(u.curso, '—') AS curso,
            u.role AS perfil,
            COUNT(CASE WHEN e.status='ativo' THEN 1 END) AS emprestimos_ativos,
            COUNT(CASE WHEN e.status='devolvido' THEN 1 END) AS devolvidos
        FROM usuarios u
        LEFT JOIN emprestimos e ON e.aluno_id = u.id
        GROUP BY u.id
        ORDER BY u.role, u.nome
    """, conn)
    conn.close()
    return df


def listar_livros_status():
    """Lista livros com quantidade emprestada e disponível."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT
            l.id,
            l.titulo,
            l.autor,
            l.area,
            l.quantidade_total      AS total,
            l.quantidade_disponivel AS disponivel,
            (l.quantidade_total - l.quantidade_disponivel) AS emprestados,
            COALESCE(
                (SELECT COUNT(*) FROM emprestimos e
                 WHERE e.livro_id = l.id AND e.status='devolvido'), 0
            ) AS devolucoes
        FROM livros l
        ORDER BY l.titulo
    """, conn)
    conn.close()
    return df


def listar_emprestimos_completos():
    """Histórico completo (ativos + devolvidos) com nome do aluno e título."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT
            e.id,
            u.nome          AS aluno,
            u.email         AS email,
            l.titulo        AS livro,
            e.data_emprestimo AS emprestado_em,
            COALESCE(e.data_devolucao, '—') AS devolvido_em,
            e.status
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        ORDER BY e.data_emprestimo DESC
    """, conn)
    conn.close()
    return df


def listar_devolvidos():
    """Apenas empréstimos devolvidos."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT
            e.id,
            u.nome   AS aluno,
            u.email  AS email,
            l.titulo AS livro,
            e.data_emprestimo AS emprestado_em,
            e.data_devolucao  AS devolvido_em
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        WHERE e.status='devolvido'
        ORDER BY e.data_devolucao DESC
    """, conn)
    conn.close()
    return df


def listar_emprestados():
    """Apenas empréstimos ativos."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT
            e.id,
            u.nome   AS aluno,
            u.email  AS email,
            l.titulo AS livro,
            e.data_emprestimo AS emprestado_em
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        WHERE e.status='ativo'
        ORDER BY e.data_emprestimo DESC
    """, conn)
    conn.close()
    return df

print("✅ Funções do dashboard carregadas")

✅ Funções do dashboard carregadas


In [15]:
# @title 🔐 Interface do Administrador (corrigida, com empréstimos completos)
import gradio as gr
import sqlite3
import pandas as pd


# ---------- FUNÇÕES DE DADOS ----------
def resumo_geral():
    conn = sqlite3.connect(DB_PATH)
    q = lambda sql: pd.read_sql(sql, conn).iloc[0, 0]
    alunos = q("SELECT COUNT(*) FROM usuarios WHERE role='aluno'")
    livros = q("SELECT COUNT(*) FROM livros")
    total_ex = q("SELECT COALESCE(SUM(quantidade_total),0) FROM livros")
    disp = q("SELECT COALESCE(SUM(quantidade_disponivel),0) FROM livros")
    ativos = q("SELECT COUNT(*) FROM emprestimos WHERE status='ativo'")
    devolv = q("SELECT COUNT(*) FROM emprestimos WHERE status='devolvido'")
    conn.close()
    return f"""
### 📈 Resumo Geral do Sistema

| Métrica | Valor |
|---|---|
| 👥 Alunos cadastrados | **{alunos}** |
| 📚 Títulos no acervo | **{livros}** |
| 📦 Exemplares totais | **{total_ex}** |
| ✅ Exemplares disponíveis | **{disp}** |
| 📕 Empréstimos ativos | **{ativos}** |
| 📗 Empréstimos devolvidos | **{devolv}** |
"""


def usuarios_df():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT u.id, u.nome, u.email,
               COALESCE(u.curso, '—') AS curso,
               u.role AS perfil,
               COUNT(CASE WHEN e.status='ativo' THEN 1 END) AS emprestimos_ativos,
               COUNT(CASE WHEN e.status='devolvido' THEN 1 END) AS devolvidos
        FROM usuarios u
        LEFT JOIN emprestimos e ON e.aluno_id = u.id
        GROUP BY u.id
        ORDER BY u.role, u.nome
    """, conn)
    conn.close()
    return df


def livros_df():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT l.id, l.titulo, l.autor, l.area,
               l.quantidade_total AS total,
               l.quantidade_disponivel AS disponivel,
               (l.quantidade_total - l.quantidade_disponivel) AS emprestados
        FROM livros l
        ORDER BY l.titulo
    """, conn)
    conn.close()
    return df


def emprestimos_ativos_df():
    """Quem está com qual livro agora."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT e.id, u.nome AS aluno, u.email AS email,
               l.titulo AS livro, l.autor AS autor,
               e.data_emprestimo AS emprestado_em
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        WHERE e.status='ativo'
        ORDER BY e.data_emprestimo DESC
    """, conn)
    conn.close()
    return df


def emprestimos_devolvidos_df():
    """Quem devolveu o quê e quando."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT e.id, u.nome AS aluno, u.email AS email,
               l.titulo AS livro,
               e.data_emprestimo AS emprestado_em,
               e.data_devolucao  AS devolvido_em
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        WHERE e.status='devolvido'
        ORDER BY e.data_devolucao DESC
    """, conn)
    conn.close()
    return df


def historico_completo_df():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT e.id, u.nome AS aluno, u.email AS email,
               l.titulo AS livro,
               e.data_emprestimo AS emprestado_em,
               COALESCE(e.data_devolucao, '—') AS devolvido_em,
               e.status AS status
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        ORDER BY e.data_emprestimo DESC
    """, conn)
    conn.close()
    return df


def consultar_por_aluno(termo):
    if not termo:
        return pd.DataFrame()
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT u.nome AS aluno, u.email, l.titulo AS livro,
               e.data_emprestimo AS emprestado_em,
               COALESCE(e.data_devolucao, '—') AS devolvido_em,
               e.status
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        WHERE u.nome LIKE ? OR u.email LIKE ?
        ORDER BY e.status, e.data_emprestimo DESC
    """, conn, params=(f"%{termo}%", f"%{termo}%"))
    conn.close()
    return df


def consultar_por_livro(termo):
    if not termo:
        return pd.DataFrame()
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql("""
        SELECT l.titulo AS livro, u.nome AS aluno, u.email, u.curso,
               e.data_emprestimo AS emprestado_em,
               COALESCE(e.data_devolucao, '—') AS devolvido_em,
               e.status
        FROM emprestimos e
        JOIN usuarios u ON u.id = e.aluno_id
        JOIN livros   l ON l.id = e.livro_id
        WHERE l.titulo LIKE ?
        ORDER BY e.status, e.data_emprestimo DESC
    """, conn, params=(f"%{termo}%",))
    conn.close()
    return df


def fazer_login_admin(email, senha):
    user = autenticar(email, senha)
    if not user or user['role'] != 'admin':
        return (
            None,
            "❌ Acesso negado.",
            gr.update(visible=True),
            gr.update(visible=False),
            "", pd.DataFrame(), pd.DataFrame(),
            pd.DataFrame(), pd.DataFrame(), pd.DataFrame(),
        )
    return (
        user['id'],
        f"✅ Bem-vindo, **{user['nome']}**",
        gr.update(visible=False),
        gr.update(visible=True),
        resumo_geral(),
        usuarios_df(),
        livros_df(),
        emprestimos_ativos_df(),
        emprestimos_devolvidos_df(),
        historico_completo_df(),
    )


def atualizar_dashboard():
    return (
        resumo_geral(),
        usuarios_df(),
        livros_df(),
        emprestimos_ativos_df(),
        emprestimos_devolvidos_df(),
        historico_completo_df(),
    )


# ---------- INTERFACE ----------
with gr.Blocks(theme=gr.themes.Soft(), title="BiblioRec — Admin") as app_admin:
    gr.Markdown("# 🔐 BiblioRec — Painel Administrativo")
    gr.Markdown("*Login:* `admin@bibliorec.com` / `admin123`")

    st_admin = gr.State(None)

    # ========== LOGIN ==========
    with gr.Group(visible=True) as grupo_auth_admin:
        with gr.Row():
            email = gr.Textbox(label="E-mail", value="admin@bibliorec.com")
            senha = gr.Textbox(label="Senha", type="password", value="admin123")
        btn_login = gr.Button("Entrar", variant="primary")
        status = gr.Markdown()

    # ========== DASHBOARD ==========
    with gr.Group(visible=False) as grupo_dash:
        btn_atualizar = gr.Button("🔄 Atualizar dados", variant="secondary")
        resumo_md = gr.Markdown()

        with gr.Tabs():
            with gr.Tab("👥 Usuários cadastrados"):
                gr.Markdown("E-mails e cursos de todos os cadastrados:")
                tab_usuarios = gr.Dataframe(interactive=False)

            with gr.Tab("📚 Livros — Estoque"):
                gr.Markdown("Total, emprestados e disponíveis por título:")
                tab_livros = gr.Dataframe(interactive=False)

            with gr.Tab("📕 Empréstimos ativos"):
                gr.Markdown("**Quem está com qual livro agora:**")
                tab_emprestados = gr.Dataframe(interactive=False)

            with gr.Tab("📗 Devolvidos"):
                gr.Markdown("**Quem devolveu, o quê e quando:**")
                tab_devolvidos = gr.Dataframe(interactive=False)

            with gr.Tab("📋 Histórico completo"):
                gr.Markdown("Todos os empréstimos (ativos + devolvidos):")
                tab_completo = gr.Dataframe(interactive=False)

            with gr.Tab("🔍 Consultar aluno/livro"):
                gr.Markdown("### 👤 O que um aluno pegou?")
                with gr.Row():
                    busca_aluno = gr.Textbox(label="Nome ou e-mail", placeholder="ana@teste.com")
                    btn_busca_aluno = gr.Button("Consultar", variant="primary")
                tab_busca_aluno = gr.Dataframe(interactive=False)

                gr.Markdown("---")
                gr.Markdown("### 📖 Quem está com um livro?")
                with gr.Row():
                    busca_livro = gr.Textbox(label="Título do livro", placeholder="Algoritmos")
                    btn_busca_livro = gr.Button("Consultar", variant="primary")
                tab_busca_livro = gr.Dataframe(interactive=False)

            with gr.Tab("➕ Cadastrar livro"):
                with gr.Row():
                    t = gr.Textbox(label="Título")
                    a = gr.Textbox(label="Autor")
                with gr.Row():
                    ar = gr.Textbox(label="Área")
                    isbn = gr.Textbox(label="ISBN")
                with gr.Row():
                    kw = gr.Textbox(label="Palavras-chave")
                    qtd = gr.Number(label="Quantidade", value=1, precision=0)
                btn_cad = gr.Button("Cadastrar", variant="primary")
                saida_cad = gr.Markdown()

            with gr.Tab("🔎 Buscar livro"):
                termo = gr.Textbox(label="Título, autor, área ou palavra-chave")
                btn_busca = gr.Button("Buscar")
                resultado_busca = gr.Dataframe(interactive=False)

    # ========== LIGAÇÕES ==========
    btn_login.click(
        fazer_login_admin,
        inputs=[email, senha],
        outputs=[st_admin, status, grupo_auth_admin, grupo_dash,
                 resumo_md, tab_usuarios, tab_livros,
                 tab_emprestados, tab_devolvidos, tab_completo],
    )

    btn_atualizar.click(
        atualizar_dashboard,
        outputs=[resumo_md, tab_usuarios, tab_livros,
                 tab_emprestados, tab_devolvidos, tab_completo],
    )

    btn_busca_aluno.click(consultar_por_aluno, inputs=busca_aluno, outputs=tab_busca_aluno)
    btn_busca_livro.click(consultar_por_livro, inputs=busca_livro, outputs=tab_busca_livro)

    btn_cad.click(
        cadastrar_livro,
        inputs=[t, a, ar, isbn, kw, qtd],
        outputs=saida_cad,
    ).then(
        lambda: atualizar_dashboard()[1:],
        outputs=[tab_usuarios, tab_livros, tab_emprestados,
                 tab_devolvidos, tab_completo],
    )

    btn_busca.click(buscar_livros, inputs=termo, outputs=resultado_busca)

In [11]:
# @title 🎓 Executar interface do ALUNO
app_aluno.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8a9b8c26f9babf563.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
# @title 🔐 Executar interface do ADMIN (rode em outra célula)
app_admin.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://455723efe9a8675f02.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
